# BM25 Keyword Search with pg_textsearch

Vector search (pgvector) is great at *semantic* similarity, but it can miss exact keywords, rare terms, product codes, or names - cases where the wording itself matters. **BM25** is the classic keyword ranking function that search engines use: it scores a document by how well its terms match the query, weighting rare terms more and dampening very frequent ones.

[`pg_textsearch`](https://github.com/timescale/pg_textsearch) adds BM25 ranking to PostgreSQL as a native index. This notebook searches the **document chunks already ingested** in `1_standard_rag/4_ocr_chunk_store_pgvector.ipynb`, which built a BM25 index named **`bm25_chunks_idx`** over the chunk text. We read that index directly with `psycopg` (BM25 is plain SQL - there is no LangChain wrapper for it yet).

Prerequisites:

- Run `1_standard_rag/4_ocr_chunk_store_pgvector.ipynb` first - it ingests the PDF and creates `bm25_chunks_idx`.
- The database must have `pg_textsearch` installed and preloaded (see `1_standard_rag/1_install_pgvector`).

## Dependencies

Only `psycopg` (the PostgreSQL driver) and `python-dotenv` are needed, both already in `../requirements.txt`:

```bash
pip install -r requirements.txt
```

## Configuration

Reuse the shared **`day2skk/var.env`** file (one folder up: `../var.env`), the same one the standard-RAG notebooks use - no separate `.env` per notebook. This runs inside the cluster, so `PG_HOST` is the `pgvector` service name. We search the same collection notebook 4 ingested into, `rag_documents`.

```dotenv
PG_HOST=pgvector
PG_PORT=5432
PG_USER=raguser
PG_PASSWORD=change-me-please
PG_DB=ragdb
```

In [1]:
import os
import psycopg
from dotenv import load_dotenv

load_dotenv("../var.env")   # shared env file at day2skk/var.env

PG_HOST = os.environ.get("PG_HOST", "pgvector")
PG_PORT = os.environ.get("PG_PORT", "5432")
PG_USER = os.environ.get("PG_USER", "raguser")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "change-me-please")
PG_DB = os.environ.get("PG_DB", "ragdb")

COLLECTION_NAME = "rag_documents"   # the collection ingested in notebook 4
INDEX_NAME = "bm25_chunks_idx"      # the BM25 index created in notebook 4

conn = psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
)
conn.autocommit = True
print("connected to", f"{PG_HOST}:{PG_PORT}/{PG_DB}")

connected to 103.125.91.67:5432/ragdb


## Check the extension, index, and chunks

BM25 search needs three things, all produced by notebook 4 (plus the `1_install_pgvector` step): the `pg_textsearch` extension, the `bm25_chunks_idx` index, and some ingested chunks in the `rag_documents` collection. We confirm each and grab the collection's id, so our query only scores this collection's rows.

In [2]:
with conn.cursor() as cur:
    # 1. extension installed?
    cur.execute("SELECT extversion FROM pg_extension WHERE extname = 'pg_textsearch';")
    ext = cur.fetchone()
    if ext is None:
        raise RuntimeError("pg_textsearch is not installed. See 1_standard_rag/1_install_pgvector.")

    # 2. the BM25 index from notebook 4 exists?
    cur.execute("SELECT 1 FROM pg_indexes WHERE indexname = %s;", (INDEX_NAME,))
    if cur.fetchone() is None:
        raise RuntimeError(
            f"Index {INDEX_NAME!r} not found. Run 1_standard_rag/4_ocr_chunk_store_pgvector.ipynb first."
        )

    # 3. our collection and how many chunks it has
    cur.execute("SELECT uuid FROM langchain_pg_collection WHERE name = %s;", (COLLECTION_NAME,))
    row = cur.fetchone()
    if row is None:
        raise RuntimeError(f"Collection {COLLECTION_NAME!r} not found. Run notebook 4 first.")
    COLLECTION_ID = row[0]

    cur.execute(
        "SELECT count(*) FROM langchain_pg_embedding WHERE collection_id = %s;",
        (COLLECTION_ID,),
    )
    n_chunks = cur.fetchone()[0]

print("pg_textsearch version:", ext[0])
print(f"index {INDEX_NAME!r}: found")
print(f"collection {COLLECTION_NAME!r}: {n_chunks} chunks")

pg_textsearch version: 1.4.0-dev
index 'bm25_chunks_idx': found
collection 'rag_documents': 18 chunks


## Search with BM25 over the ingested chunks

`langchain-postgres` stored each chunk's text in the `document` column of its `langchain_pg_embedding` table; notebook 4 put a BM25 index on it. We score each chunk with the **`<@>`** operator, explicitly reading the `bm25_chunks_idx` index via **`to_bm25query('terms', 'bm25_chunks_idx')`**, and filter to our collection.

One quirk: BM25 scores here are **negative**, and **lower (more negative) means a better match**, so `ORDER BY score` (ascending, the default) returns the best matches first.

In [5]:
def bm25_search(query: str, k: int = 5):
    sql = (
        "SELECT document, document <@> to_bm25query(%(q)s, %(idx)s) AS score "
        "FROM langchain_pg_embedding "
        "WHERE collection_id = %(cid)s "
        "ORDER BY score "      # ascending: most-negative (best) first
        "LIMIT %(k)s;"
    )
    with conn.cursor() as cur:
        cur.execute(sql, {"q": query, "idx": INDEX_NAME, "cid": COLLECTION_ID, "k": k})
        return cur.fetchall()


query = "What is Cloudeka?"
print(f"BM25 results for: {query!r}\n")
for content, score in bm25_search(query):
    print(f"  score={score:.4f}  {content[:80]!r}")

BM25 results for: 'What is Cloudeka?'

  score=-1.4491  'Service Portal Cloudeka\nCloudeka is a Cloud Computing platform that provides var'
  score=-1.1379  'Introduction\nTo create Deka LLM in the Service Portal Cloudeka, you need to know'
  score=-1.1050  'Introduction\nPrevious Location Next Create Deka VPN\n[Guidance for Enterprise Clo'
  score=-1.1050  'Introduction\n[Previous Delete CDN](https://docs.cloudeka.ai/deka-cdn/delete-cdn)'
  score=-0.9860  'Introduction\nDeka Box is S3 Browser compatible storage objects spread across thr'


### Keyword precision

BM25 ranks the chunks that contain the exact query terms highest. The best queries to try depend on what is in *your* ingested PDF - change the terms below to words that actually appear in it.

In [4]:
# Adjust these to terms present in your document.
for query in ["Deka Box", "storage", "hello"]:
    print(f"\nBM25 results for: {query!r}")
    for content, score in bm25_search(query, k=3):
        print(f"  score={score:.4f}  {content[:80]!r}")


BM25 results for: 'Deka Box'
  score=-3.0804  'Introduction\nDeka Box is S3 Browser compatible storage objects spread across thr'
  score=-2.8796  '2. Postpaid\nThe Postpaid type is used for companies and registers using the comp'
  score=-0.6211  'Endpoint Deka LLM\nCurrently Deka LLM has the following model categories:'

BM25 results for: 'storage'
  score=-1.9517  'Introduction\nDeka Box is S3 Browser compatible storage objects spread across thr'
  score=-1.2950  'Service Portal Cloudeka\nCloudeka is a Cloud Computing platform that provides var'
  score=-1.1485  'Introduction\n[Previous](https://docs.cloudeka.ai/deka-notebook/how-to-create-a-s'

BM25 results for: 'hello'
  score=0.0000  '1. Prepaid\nFor the Prepaid type, it is used for personal needs that use personal'
  score=0.0000  '2. Postpaid\nThe Postpaid type is used for companies and registers using the comp'
  score=0.0000  'Service Portal Cloudeka\nCloudeka is a Cloud Computing platform that provides var'


## How this fits hybrid RAG

Dense (pgvector) and sparse (BM25) retrieval have complementary strengths:

- **Vector search** catches paraphrases and meaning ("car" matches "automobile").
- **BM25** nails exact terms, names, codes, and rare words that embeddings blur together.

Both read the **same chunks** here: pgvector uses the embeddings, BM25 uses the `bm25_chunks_idx` index over the same `document` column. **Hybrid retrieval** runs both and fuses the two ranked lists - commonly with *Reciprocal Rank Fusion (RRF)* - so a chunk that scores well on either signal surfaces. The next notebook (`2_hybrid_search_rrf.ipynb`) does exactly that.

## Recap

- **`pg_textsearch`** brings BM25 keyword ranking into PostgreSQL as a native index.
- This notebook **reads the `bm25_chunks_idx` index** built during ingestion (notebook 4) over the chunks in the `rag_documents` collection - no separate table or re-indexing.
- Search with the **`<@>`** operator, naming the index via **`to_bm25query('terms', 'bm25_chunks_idx')`** and filtering by `collection_id`; scores are **negative**, so `ORDER BY score` returns the best matches first.
- BM25 complements vector search; fusing both gives **hybrid retrieval**, the focus of the next notebook.